In [ ]:
"""
Excluding Experimental+IN718 literature
SVM (SVC) Training + Prediction Script
Python 3.10
Dependencies: pandas, numpy, scikit-learn, joblib
"""

import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

# -------------------------------
# 1. Load dataset
# -------------------------------
dataset_path = r"C:\Users\gurra\Downloads\Excluding_30_IN718.csv"
dataset = pd.read_csv(dataset_path)

# -------------------------------
# 2. Features & Target
# -------------------------------
features = [
    'power','speed','thickness','dia','Solidt','Sdensity','Sspheat','Sthercondu',
    'Liquidt','Ldensity','LSpheat','Lthercondu','Lsurfacet','Lviscosity',
    'Lfusion','dsigma','Absorptivity','Lvapor'
]
X = dataset[features]
y = dataset['defect']

# -------------------------------
# 3. Scale features
# -------------------------------
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (here using full dataset for training if desired)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.15, random_state=42)

# -------------------------------
# 4. SVM + GridSearchCV
# -------------------------------
param_grid = {
    "C": [0.1, 1, 10, 50, 100, 150, 180, 200, 250],
    "kernel": ["linear", "rbf", "poly", "sigmoid"],
    "gamma": ["scale", "auto"]
}

svm_model = SVC(probability=True, random_state=42)

grid = GridSearchCV(
    estimator=svm_model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

# -------------------------------
# 5. Evaluate model
# -------------------------------
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print("\nTrain Accuracy:", accuracy_score(y_train, y_train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
print("\nConfusion Matrix (Test Set):\n", confusion_matrix(y_test, y_test_pred))
print("\nClassification Report (Test Set):\n", classification_report(y_test, y_test_pred, labels=[0,1,2,3]))

# -------------------------------
# 6. Save model & scaler
# -------------------------------
joblib.dump(best_model, "SVC_model.pkl")
joblib.dump(scaler, "SVC_scaler.pkl")

# -------------------------------
# 7. Predict new alloy dataset
# -------------------------------
new_data_path = r"C:\Users\gurra\Downloads\final 5-800w power speed 718 alloy predictions.csv"
new_data = pd.read_csv(new_data_path)

# Scale features
X_new_scaled = scaler.transform(new_data[features])

# Predictions & probabilities
y_new_pred = best_model.predict(X_new_scaled)
y_new_proba = best_model.predict_proba(X_new_scaled)

# Map numeric predictions to labels
class_map = {0: 'Good', 1: 'Balling', 2: 'Lack of fusion', 3: 'Keyholing'}
new_data['Predicted_class'] = y_new_pred
new_data['Predicted_Label'] = new_data['Predicted_class'].map(class_map)

# Add probability columns
proba_df = pd.DataFrame(
    y_new_proba,
    columns=[f"Prob_{class_map[c]}" for c in sorted(class_map.keys())]
)
new_data = pd.concat([new_data, proba_df], axis=1)

# Save predictions
output_file = r"C:\Users\gurra\Downloads\EE_718_SVC_predictions.csv"
new_data.to_csv(output_file, index=False)
print(f"\nPredictions saved to: {output_file}")
joblib.dump(best_model, r"C:\Users\gurra\Downloads\SVC_model.pkl")
joblib.dump(scaler, r"C:\Users\gurra\Downloads\SVC_scaler.pkl")


Fitting 5 folds for each of 72 candidates, totalling 360 fits

Best Parameters: {'C': 250, 'gamma': 'auto', 'kernel': 'rbf'}
Best CV Accuracy: 0.7876095850379895

Train Accuracy: 0.910958904109589
Test Accuracy: 0.8653846153846154

Confusion Matrix (Test Set):
 [[ 9  1  1  0]
 [ 2 12  1  0]
 [ 1  0 14  0]
 [ 1  0  0 10]]

Classification Report (Test Set):
               precision    recall  f1-score   support

           0       0.69      0.82      0.75        11
           1       0.92      0.80      0.86        15
           2       0.88      0.93      0.90        15
           3       1.00      0.91      0.95        11

    accuracy                           0.87        52
   macro avg       0.87      0.87      0.87        52
weighted avg       0.88      0.87      0.87        52


Predictions saved to: C:\Users\gurra\Downloads\EE_SS_SVC_predictions.csv


['C:\\Users\\gurra\\Downloads\\SVC_scaler.pkl']